# Spectroscopy Crystal Alignment

This notebooks shows the alignment procedure for crystals in spectroscopy.
The output of this procedure is a $UB$ matrix for a given sample.

<div class="alert alert-warning">

**Attention**

The current implementation in ScippNeutron was designed for Spectroscopy.
It uses a convention for the U matrix from that field.
This differs from single crystal diffraction.
Support for that technique has not been implemented yet.

</div>

We will use results from a simulation of a sample with known lattice parameters and a set of measured Bragg peaks and their Miller indices.

In [ ]:
import numpy as np
import scipp as sc

from scippneutron.spectroscopy.crystal import alignment, lattice
from scippneutron.spectroscopy.goniometer import Goniometer, GoniometerAxis
from scippneutron.spectroscopy.geometry import ScatteringPlaneMiller

## Experiment Setup

We begin by describing the experimental setup.

First, we define a `Goniometer` object to encode the rotation axes.
Here, we set the limits to infinity for simplicity; in a real experiment, you should use them to check whether the rotations you compute are accessible with your hardware.

In [ ]:
inf = sc.scalar(np.inf, unit='rad')
goniometer = Goniometer(
    axes=[
        # Connects to the sample:
        GoniometerAxis(name='+x', axis="+x", min=-inf, max=inf),
        GoniometerAxis(name='-z', axis="-z", min=-inf, max=inf),
        # Connects to the lab frame:
        GoniometerAxis(name='+y', axis="+y", min=-inf, max=inf),
    ]
)
_ = goniometer.plot()

Next, define the incident energy used in this experiment:

In [ ]:
ei = sc.scalar(14.0, unit='meV')

And finally, we need to set some sample parameters.

The (approximately) known lattice parameters:

In [ ]:
lattice_params = lattice.LatticeParameters(
    a=sc.scalar(5.0556, unit='Å'),
    b=sc.scalar(5.0556, unit='Å'),
    c=sc.scalar(24.5980, unit='Å'),
    alpha=sc.scalar(90.0, unit='deg'),
    beta=sc.scalar(90.0, unit='deg'),
    gamma=sc.scalar(120.0, unit='deg')
)

We need a couple of peaks to perform and test the alignment procedure:

In [ ]:
# Define the scattering plane in Miller Indices:
miller_plane = ScatteringPlaneMiller.from_in_plane_vectors(
    sc.vector([0, 0, 6]),  # (optional) order s.t. normal points along +y
    sc.vector([2, 1, 0]),
)

known_q_sample = sc.vectors(
    dims=['peak'],
    unit='1/Å',
    values=[
        [0.0002, -0.0003, 1.5328],
        [2.4763, 0.2172, 0.0001],
        [1.3008, -0.6070, 0.2552],
        [2.4765, 0.2172, 1.5329],
        [-1.3014, 0.6067, 2.8098],
        [1.3008, -0.6069, 0.0001],
    ]
)
known_hkl = sc.vectors(
    dims=['peak'],
    values=[
        [0, 0, 6],
        [2, 1, 0],
        [1, 0, 1],
        [2, 1, 6],
        [-1, 0, 11],
        [1, 0, 0],
    ]
)

# We also have the goniometer angles and scattering angle
# (two theta) of the first two peaks:
known_angles1 = sc.array(dims=['angles'], values=[0.0, -5.893, 72.85], unit='deg')
known_two_theta1 = sc.scalar(-34.290, unit='deg')
known_angles2 = sc.array(dims=['angles'], values=[0.0, -5.893, -46.920], unit='deg')
known_two_theta2 = sc.scalar(-93.830, unit='deg')

## First Peak

In this section, we assume that we measured only the first peak in the list above.
We want to compute the first estimate of the $UB$ matrix from this peak.

First, we need to compute an estimate of the $B$ matrix based on the (approximate) lattice parameters:

In [ ]:
b0 = lattice.b_matrix_from_lattice_parameters(lattice_params)

Then we need to encode the first peak in a `BraggPeaks` object:

In [ ]:
first_peak = alignment.BraggPeaks.from_q_sample(
    q_sample=known_q_sample[0],
    hkl=known_hkl[0],
    # Use the goniometer to compute a rotation from the angles:
    r=goniometer.rotation_for(known_angles1),
)
# Alternatively, if it was easier to access Q in the lab frame:
# first_peak = alignment.BraggPeaks.from_q_lab(
#     q_lab=known_q_lab[0],
#     hkl=known_hkl[0],
#     r=goniometer.rotation_for(known_angles1)
# )

Now, we can compute the rotation $U$:

In [ ]:
u1 = alignment.u_from_1_peak_and_scattering_plane(
    peaks=first_peak,
    scattering_plane=miller_plane,
    b=b0,
)
u1

`u1` is encoded as a Scipp Variable with `dtype` 'rotation3' which uses quaternions instead of a matrix.
We can turn this into a matrix using the following.

In [ ]:
one = sc.spatial.linear_transform(value=np.eye(3))
np.round((u1 * one).values, 5)

The resulting UB matrix is

In [ ]:
ub1 = u1 * b0
print("Unit =", ub1.unit)
np.round(ub1.values, 5)

## Second Peak

Now we want to refine the UB matrix by measuring a second peak.
We use the second peak from the list of known peaks and initially pretend that we only know its Miller Indices and a scattering angle $2\theta$ where we want to see the peak.

### Locating the peak

We need to compute goniometer angles that place the selected peak at the given $2\theta$.
First, define the scattering plane in the sample frame:

In [ ]:
sample_plane = alignment.scattering_plane_from_one_peak(
    first_peak, ub=ub1
)
sample_plane.normal.value

Then compute the rotation:

In [ ]:
rotation_peak2 = alignment.target_rotation_for_reflection(
    hkl=known_hkl[1],
    two_theta=known_two_theta2,
    ei=ei,
    ef=ei,  # We only care about elastic scattering for locating peaks.
    ub=ub1,
    scattering_plane=sample_plane,
)

As with `u1`, `rotation_peak2` is a variable with `dtype` 'rotation3'.
We can compute goniometer angles from it like this:

In [ ]:
angles2 = goniometer.decompose_rotation(rotation_peak2)
for ax, angle in zip(goniometer.axes, angles2, strict=True):
    print(f"{ax.name} : {angle.to(unit='deg').value:+.4f} deg")

We can check that the computed angles agree with the simulation:

In [ ]:
known_angles2

### Update the UB matrix

Having rotated the peak into view, we can measure it and obtain a $Q_\text{sample}$.
This allows us to compute a better estimate for the UB matrix:

In [ ]:
first_2_peaks = alignment.BraggPeaks.from_q_sample(
    q_sample=known_q_sample[:2],
    hkl=known_hkl[:2],
    r=None,  # We no longer need the sample rotation
)

u2 = alignment.u_from_2_peaks(first_2_peaks, b0)
ub2 = u2 * b0

print("Unit =", ub2.unit)
np.round(ub2.values, 5)

We can compare the measured $Q_{\text{sample}}$ values with values computed from $(hkl)$ to check the $UB$ matrix:

In [ ]:
def compare(ub, hkl, q_sample):
    return np.round((ub * hkl * 2 * np.pi - q_sample).values, 5)

The previous matrix yields good results for the first peak but is less good for the second peak:

In [ ]:
compare(ub1, first_2_peaks.hkl, first_2_peaks.q_sample)

The new matrix yields slightly better results, but is still fare from ideal:

In [ ]:
compare(ub2, first_2_peaks.hkl, first_2_peaks.q_sample)

## Third Peak

Now we add a third peak to improve the UB matrix further.
We could compute the required goniometer angles as before, but since the procedure is identical, we skip it here for brevity.

We can compute a $UB$ matrix from the first three measured peaks like so:

In [ ]:
first_three_peaks = alignment.BraggPeaks.from_q_sample(
    q_sample=known_q_sample[:3],
    hkl=known_hkl[:3],
)
ub3 = alignment.ub_from_3_peaks(first_three_peaks)

Note that this no longer returns only $U$ but a combined $UB$ matrix.
This can yield better results because it is free to adjust $B$ as well.
We can indeed see that this new matrix reproduces the three peaks much better:

In [ ]:
compare(ub3, first_three_peaks.hkl, first_three_peaks.q_sample)

## Many Peaks

We can refine the $UB$ matrix even more by using additional peaks.
Here we use all six known peaks:

In [ ]:
many_peaks = alignment.BraggPeaks.from_q_sample(
    q_sample=known_q_sample,
    hkl=known_hkl,
)
ub_many = alignment.ub_from_many_peaks(many_peaks)

We can see that this $UB$ matrix yields more consistent results than `ub3` even tough `ub3` is better at representing the first three peaks:

In [ ]:
compare(ub_many, known_hkl, known_q_sample)

In [ ]:
compare(ub3, known_hkl, known_q_sample)